In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_auc_score
)

print(f"PyTorch version: {torch.__version__}")

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple MPS GPU")
else:
    device = torch.device("cpu")
    print("Using CPU")

torch.set_float32_matmul_precision("high")

In [ ]:
train_dir = "/Users/mithils/TDA/archive/train"
test_dir  = "/Users/mithils/TDA/archive/test"

In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

test_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

In [ ]:
full_train_dataset = datasets.ImageFolder(train_dir)
test_dataset       = datasets.ImageFolder(test_dir, transform=test_transform)

train_size = int(0.8 * len(full_train_dataset))
val_size   = len(full_train_dataset) - train_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

train_dataset.dataset.transform = train_transform
val_dataset.dataset.transform   = test_transform

print(f"Classes     : {full_train_dataset.classes}")
print(f"Train images: {train_size}")
print(f"Val   images: {val_size}")
print(f"Test  images: {len(test_dataset)}")

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=16, shuffle=False, num_workers=0, pin_memory=True)

print(f"Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

In [ ]:
model = models.resnet18(pretrained=True)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, 1)
model    = model.to(device)

print("Backbone frozen. Training FC layer only.")

In [ ]:
THRESHOLD      = 0.5
_EPOCHS  = 3
_EPOCHS  = 5    
LR_= 1e-3
LR_= 1e-5
PATIENCE       = 2    
BEST_MODEL_PATH = "best_model.pth"

criterion = nn.BCEWithLogitsLoss()
print("Config set.")

In [ ]:
def evaluate(loader):
    model.eval()
    correct    = 0
    total      = 0
    all_preds  = []
    all_labels = []
    all_probs  = []  

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            probs   = torch.sigmoid(outputs).squeeze()
            preds   = (probs > THRESHOLD).long()

            correct += (preds == labels).sum().item()
            total   += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    accuracy = correct / total
    return accuracy, all_preds, all_labels, all_probs

In [ ]:
optimizer1 = optim.Adam(model.fc.parameters(), lr=LR_)
scheduler1 = torch.optim.lr_scheduler.StepLR(optimizer1, step_size=2, gamma=0.1)

train_losses   = []
val_accuracies = []
best_val       = 0

print(" FC layer only ")

for epoch in range(_EPOCHS):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer1.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer1.step()
        total_loss += loss.item()

    val_acc, _, _, _ = evaluate(val_loader)
    scheduler1.step()

    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)
    val_accuracies.append(val_acc)

    if val_acc > best_val:
        best_val = val_acc
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"   Best model saved (val acc: {val_acc*100:.2f}%)")

    print(f"Epoch [{epoch+1}/{_EPOCHS}]  Loss: {avg_loss:.4f}  Val Acc: {val_acc*100:.2f}%")

print("\ncomplete!")

In [ ]:
for param in model.parameters():
    param.requires_grad = True

optimizer2 = optim.Adam(model.parameters(), lr=LR_)
scheduler2 = torch.optim.lr_scheduler.StepLR(optimizer2, step_size=1, gamma=0.5)

counter = 0

print(" Full fine-tuning with early stopping ")

for epoch in range(_EPOCHS):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer2.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer2.step()
        total_loss += loss.item()

    val_acc, _, _, _ = evaluate(val_loader)
    scheduler2.step()

    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)
    val_accuracies.append(val_acc)

    if val_acc > best_val:
        best_val = val_acc
        counter  = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"   Best model saved (val acc: {val_acc*100:.2f}%)")
    else:
        counter += 1
        print(f"   No improvement ({counter}/{PATIENCE})")

    print(f"Epoch [{epoch+1}/{_EPOCHS}]  Loss: {avg_loss:.4f}  Val Acc: {val_acc*100:.2f}%")

    if counter >= PATIENCE:
        print("\n Early stopping triggered — restoring best model.")
        model.load_state_dict(torch.load(BEST_MODEL_PATH))
        break

print("\ncomplete!")

In [ ]:
total_epochs = len(train_losses)
epochs_range = range(1, total_epochs + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(epochs_range, train_losses, marker='o', color='steelblue')
ax1.axvline(x=_EPOCHS + 0.5, color='red', linestyle='--', label='Fine-tune starts')
ax1.set_title("Training Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(True)

ax2.plot(epochs_range, [v*100 for v in val_accuracies], marker='o', color='green')
ax2.axvline(x=_EPOCHS + 0.5, color='red', linestyle='--', label='Fine-tune starts')
ax2.set_title("Validation Accuracy")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
test_acc, preds, labels, probs = evaluate(test_loader)

precision = precision_score(labels, preds)
recall    = recall_score(labels, preds)
f1        = f1_score(labels, preds)
auc       = roc_auc_score(labels, probs)  

print(f"Test Accuracy : {test_acc*100:.2f}%")
print(f"Precision     : {precision:.4f}")
print(f"Recall        : {recall:.4f}")
print(f"F1 Score      : {f1:.4f}")
print(f"AUC Score     : {auc:.4f}")

In [ ]:
cm   = confusion_matrix(labels, preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["FAKE", "REAL"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix — Test Set")
plt.show()

In [ ]:
def predict_image(path):
    img    = Image.open(path).convert("RGB")
    tensor = test_transform(img).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        prob = torch.sigmoid(model(tensor)).item()

    label = "REAL" if prob > THRESHOLD else "FAKE"

    confidence = prob if prob > THRESHOLD else 1 - prob

    plt.imshow(img)
    plt.title(f"Prediction: {label}  |  Confidence: {confidence:.3f}")
    plt.axis("off")
    plt.show()

    return label, confidence



In [ ]:
torch.save({
    'model_state_dict'    : model.state_dict(),
    'optimizer_state_dict': optimizer2.state_dict(),
    'test_accuracy'       : test_acc,
    'auc_score'           : auc
}, "real_vs_fake_v3_final.pth")

print(f"Saved — Accuracy: {test_acc*100:.2f}% | AUC: {auc:.4f}")